# Notebook 2: DuckDB & Vector Store Test

**Objective**: Test storage layer — table registration, schema context, vector embeddings.

**Prerequisite**: Run Notebook 1 first to generate cleaned CSVs in `data/synthetic/`

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import sys, os, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, HTML

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from hybridtablerag.storage.store import DuckDBStore
from hybridtablerag.storage.schema import build_schema_context, build_multi_table_schema_context, format_schema_for_prompt
from hybridtablerag.storage.vectors import VectorStore, get_embedding_provider, SentenceTransformerProvider

def log(msg, level="info"):
    colors = {"info": "#2563eb", "warn": "#b45309", "err": "#dc2626", "ok": "#16a34a"}
    display(HTML(f"<div style='font-family:monospace;font-size:.85rem;color:{colors.get(level,'#0f172a')}'>› {msg}</div>"))

In [ ]:
# ── Initialize DuckDB ─────────────────────────────────────────────────
log("Connecting to DuckDB…")
store = DuckDBStore(db_path="data/hybridtablerag_test.duckdb")
log(f"Connected: {store.db_path}", "ok")

# List existing tables
tables = store.list_tables()
log(f"Existing tables: {tables if tables else '(none)'}")

In [ ]:
# ── Register Test Tables ──────────────────────────────────────────────
# Load cleaned CSVs from Notebook 1 output
SYNTHETIC_DIR = Path("data/synthetic")

if not SYNTHETIC_DIR.exists():
    raise FileNotFoundError(f"Run Notebook 1 first to generate {SYNTHETIC_DIR}")

registered = []
for csv_file in SYNTHETIC_DIR.glob("*.csv"):
    table_name = csv_file.stem
    df = pd.read_csv(csv_file)
    
    log(f"Registering `{table_name}` ({len(df)} rows)…")
    store.register_dataframe(df, table_name, bts_log=[])
    registered.append(table_name)

log(f"Registered {len(registered)} table(s): {registered}", "ok")
display(Markdown("**Tables in DuckDB**:"))
display(pd.DataFrame({"table": store.list_tables()}))

In [ ]:
# ── Test Schema Context Builder ───────────────────────────────────────
log("Building schema context for LLM prompts…")

if registered:
    main_table = registered[0]
    log(f"Testing single-table context for `{main_table}`…")
    
    ctx = build_schema_context(store.conn, main_table, bts_log=[])
    
    display(Markdown(f"### Schema Context: `{ctx['table_name']}`"))
    display(Markdown(f"**Row count**: {ctx['row_count']:,}"))
    
    # Show enriched column info
    for col in ctx["columns"][:8]:  # Show first 8
        line = f"- `{col['name']}` ({col['type']})"
        if col.get("all_values"):
            vals = ", ".join(f"{v['value']}({v['count']})" for v in col["all_values"][:5])
            line += f" → values: {vals}"
        elif col.get("range"):
            r = col["range"]
            line += f" → range: {r['min']} → {r['max']}"
        display(Markdown(line))
    
    # Format for prompt
    prompt_text = format_schema_for_prompt(ctx)
    display(Markdown("**Formatted for LLM prompt**:"))
    display(Markdown(f"```\n{prompt_text}\n```"))

In [ ]:
# ── Test Multi-Table Schema Context ───────────────────────────────────
if len(registered) > 1:
    log("Building multi-table schema context…")
    
    # Fake relationships for testing
    relationships = [
        {
            "from_table": registered[1] if len(registered)>1 else registered[0],
            "from_column": "id",
            "to_table": registered[0],
            "to_column": "id",
            "type": "many_to_one"
        }
    ] if len(registered) > 1 else []
    
    multi_ctx = build_multi_table_schema_context(
        store.conn, 
        registered, 
        relationships, 
        bts_log=[]
    )
    
    display(Markdown("### Multi-Table Schema Context"))
    display(Markdown(f"```json\n{json.dumps(multi_ctx, indent=2, default=str)[:1000]}...\n```"))

In [ ]:
# ── Test Vector Store (Local Embeddings) ──────────────────────────────
log("Testing vector embeddings with SentenceTransformer…")

try:
    # Use lightweight local model
    provider = SentenceTransformerProvider(model_name="all-MiniLM-L6-v2")
    log(f"Loaded embedding model (dim={provider.dimension})", "ok")
    
    vector_store = VectorStore(store.conn, provider)
    vector_store.setup()  # Install/load DuckDB vss extension
    log("DuckDB vss extension loaded", "ok")
    
    # Find text columns in main table
    if registered:
        main_table = registered[0]
        schema = store.get_table_schema(main_table)
        text_cols = [c["column_name"] for c in schema if c["data_type"] in ("VARCHAR", "TEXT")]
        
        if text_cols:
            log(f"Embedding text columns: {text_cols}")
            vector_store.embed_table(
                table_name=main_table,
                text_columns=text_cols[:2],  # Limit for speed
                pk_column=schema[0]["column_name"],  # First col as PK
                bts_log=[]
            )
            log("Embeddings created", "ok")
            
            # Test semantic search
            log("Testing semantic search…")
            results = vector_store.search(
                query="test search query",  # Replace with domain-specific query
                table_name=main_table,
                top_k=5
            )
            
            display(Markdown(f"### Vector Search Results (top 5)"))
            display(results.head() if not results.empty else Markdown("*No results*"))
        else:
            log("No text columns found for embedding", "warn")
            
except Exception as e:
    log(f"Vector test failed: {e}", "err")
    import traceback
    display(Markdown(f"```\n{traceback.format_exc()}\n```"))

In [ ]:
# ── Test Raw SQL Execution ────────────────────────────────────────────
log("Testing raw SQL execution…")

if registered:
    test_sql = f"SELECT COUNT(*) as row_count FROM {registered[0]}"
    result = store.execute_query(test_sql)
    display(Markdown(f"**Query**: `{test_sql}`"))
    display(result)
    log("SQL execution works", "ok")

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────
store.close()
log("DuckDB connection closed", "ok")

display(Markdown("## Storage Layer Test Complete"))
display(Markdown("**Next**: Run Notebook 3 to test LLM + orchestrator layer"))